In [1]:
import torch
if torch.cuda.is_available():
    print("CUDA")
else:
    print("CPU")

CUDA


In [2]:
import huggingface_hub
from datasets import load_dataset
from transformers import AutoTokenizer , Trainer
ds = load_dataset("SetFit/sst2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/378 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

dev.jsonl:   0%|          | 0.00/136k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/281k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6920 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [5]:
ds.column_names

{'train': ['text', 'label', 'label_text'],
 'validation': ['text', 'label', 'label_text'],
 'test': ['text', 'label', 'label_text']}

In [6]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
training_data = ds["train"]
def tokenize(batch):
    return tokenizer(
        batch["text"], padding="max_length", truncation=True, max_length=512
    )
training_data = training_data.map(tokenize, batched=True, remove_columns=["text", "label_text"])
training_data.set_format("torch", columns=["input_ids", "attention_mask", "label"])
testing_data = ds["test"].map(tokenize, batched=True, remove_columns=["text", "label_text"])
testing_data.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/6920 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [7]:
##training the model on the dataset
from torch import optim
import torch
from transformers import AutoModelForSequenceClassification
from torch.utils.data import DataLoader
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
EPOCHS = 3 
optimizer = optim.AdamW(model.parameters(), lr=5e-5)
model.to("cuda")
for epoch in range(EPOCHS):
    for batch in DataLoader(training_data,batch_size=16,shuffle=True):
        input_ids = batch["input_ids"].to("cuda")
        attention_mask = batch["attention_mask"].to("cuda")
        labels = batch["label"].to("cuda")
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
model.save_pretrained("./my_sentiment_model")
tokenizer.save_pretrained("./my_sentiment_model")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


('./my_sentiment_model/tokenizer_config.json',
 './my_sentiment_model/special_tokens_map.json',
 './my_sentiment_model/vocab.txt',
 './my_sentiment_model/added_tokens.json',
 './my_sentiment_model/tokenizer.json')

In [8]:
from google.colab import drive
drive.mount('/content/drive')

# Then, when saving the model, just point the path to your Drive!
model.save_pretrained("/content/drive/MyDrive/my_sentiment_model")
tokenizer.save_pretrained("/content/drive/MyDrive/my_sentiment_model")


Mounted at /content/drive


('/content/drive/MyDrive/my_sentiment_model/tokenizer_config.json',
 '/content/drive/MyDrive/my_sentiment_model/special_tokens_map.json',
 '/content/drive/MyDrive/my_sentiment_model/vocab.txt',
 '/content/drive/MyDrive/my_sentiment_model/added_tokens.json',
 '/content/drive/MyDrive/my_sentiment_model/tokenizer.json')